# Shadow Limb LSTM — Trajectory Prediction Results

This notebook evaluates the trained ankle trajectory model.

**Prerequisites:** Run `python -m src.trajectory.train` first to generate `models/shadow_limb_lstm.pt`.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.trajectory import config
from src.trajectory.model import build_model
from src.trajectory.dataset import build_subject_data, Normalizer, GaitWindowDataset, discover_trials
from src.trajectory.evaluate import (
    compute_rmse, compute_r2, compute_mae,
    per_terrain_evaluation, profile_latency,
    plot_gait_overlay, plot_error_distribution,
)
from torch.utils.data import DataLoader

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load Trained Model

In [ ]:
checkpoint = torch.load(config.MODEL_PATH, map_location=device, weights_only=False)
ckpt_cfg = checkpoint["config"]
print(f"Model type: {ckpt_cfg.get('mode', 'single')}")
print(f"Trained for {checkpoint['epoch']} epochs")
print(f"Best val RMSE: {checkpoint['val_rmse']:.4f} deg")

model = build_model(mode=ckpt_cfg.get("mode", "single")).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Parameters: {model.count_parameters():,}")

norm_path = os.path.join(config.MODEL_DIR, "normalizer.pkl")
normalizer = Normalizer().load(norm_path)

## 2. Training Curves

In [ ]:
history_path = os.path.join(config.MODEL_DIR, "training_history_trajectory.json")
with open(history_path) as f:
    history = json.load(f)

epochs = range(1, len(history["train"]) + 1)
train_rmse = [e["rmse"] for e in history["train"]]
val_rmse = [e["rmse"] for e in history["val"]]
train_r2 = [e["r2"] for e in history["train"]]
val_r2 = [e["r2"] for e in history["val"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, train_rmse, label="Train", linewidth=2)
axes[0].plot(epochs, val_rmse, label="Validation", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("RMSE (deg)")
axes[0].set_title("RMSE Over Training")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, train_r2, label="Train", linewidth=2)
axes[1].plot(epochs, val_r2, label="Validation", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("R²")
axes[1].set_title("R-Squared Over Training")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal test metrics: {history.get('test', 'not available')}")

## 3. Test Set — Overall Metrics

In [ ]:
X_test, y_test, meta_test = build_subject_data(config.TEST_SUBJECTS, verbose=False)
X_test_norm = normalizer.transform(X_test)
ds = GaitWindowDataset(X_test_norm, y_test)
loader = DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=False)

all_preds, all_targets = [], []
with torch.no_grad():
    for X_b, y_b in loader:
        p = model(X_b.to(device)).cpu().numpy()
        all_preds.append(p.flatten())
        all_targets.append(y_b.numpy().flatten())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

print(f"Test RMSE: {compute_rmse(all_preds, all_targets):.4f} deg")
print(f"Test R²:   {compute_r2(all_preds, all_targets):.4f}")
print(f"Test MAE:  {compute_mae(all_preds, all_targets):.4f} deg")
print(f"N windows: {len(all_preds):,}")

## 4. Per-Terrain Breakdown

In [ ]:
terrain_res = per_terrain_evaluation(
    model, config.TEST_SUBJECTS, normalizer, device
)

terrains = list(terrain_res.keys())
rmses = [terrain_res[t]["rmse"] for t in terrains]
r2s = [terrain_res[t]["r2"] for t in terrains]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(terrains, rmses, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].set_xlabel("RMSE (deg)")
axes[0].set_title("Per-Terrain RMSE")
for i, v in enumerate(rmses):
    axes[0].text(v + 0.05, i, f"{v:.2f}", va="center")

axes[1].barh(terrains, r2s, color="steelblue", alpha=0.8, edgecolor="white")
axes[1].set_xlabel("R²")
axes[1].set_title("Per-Terrain R-Squared")
for i, v in enumerate(r2s):
    axes[1].text(v + 0.01, i, f"{v:.3f}", va="center")

plt.tight_layout()
plt.show()

for t, m in terrain_res.items():
    print(f"  {t:15s}  RMSE={m['rmse']:.3f}  R²={m['r2']:.3f}  MAE={m['mae']:.3f}  n={m['n_windows']}")

## 5. Gait Cycle Overlay — Predicted vs. Ground Truth

In [ ]:
# Pick a trial from the test subjects to visualize
test_trials = discover_trials(config.CAMARGO_DIR, config.TEST_SUBJECTS)

if test_trials:
    # Use the first available trial
    trial = test_trials[0]
    print(f"Plotting: {trial['subject']} / {trial['condition']}")
    plot_gait_overlay(model, trial["path"], normalizer, device, n_cycles=4)
else:
    print("No test trials found. Check that the dataset is downloaded.")

## 6. Error Distribution

In [ ]:
plot_error_distribution(all_preds, all_targets)

## 7. Latency Profiling

Measures single-window inference time to verify real-time feasibility.
Target: < 20ms per window (for 200ms input windows).

In [ ]:
latency = profile_latency(model, device, n_iterations=1000)

print(f"Inference latency ({latency['device']}):")
print(f"  Mean:  {latency['mean_ms']:.2f} ms")
print(f"  Std:   {latency['std_ms']:.2f} ms")
print(f"  P95:   {latency['p95_ms']:.2f} ms")
print(f"  P99:   {latency['p99_ms']:.2f} ms")
print(f"  Min:   {latency['min_ms']:.2f} ms")
print(f"  Max:   {latency['max_ms']:.2f} ms")

if latency["p95_ms"] < 20:
    print("\nReal-time feasible: P95 < 20ms threshold.")
else:
    print(f"\nWarning: P95 latency ({latency['p95_ms']:.1f}ms) exceeds 20ms threshold.")
    print("Consider model quantization or switching to a smaller architecture.")

## 8. Per-Subject Generalization

Evaluate how well the model generalizes to each individual test subject.

In [ ]:
subject_metrics = {}
for subj in config.TEST_SUBJECTS:
    try:
        X_s, y_s, _ = build_subject_data([subj], verbose=False)
    except RuntimeError:
        continue
    X_s = normalizer.transform(X_s)
    ds_s = GaitWindowDataset(X_s, y_s)
    loader_s = DataLoader(ds_s, batch_size=config.BATCH_SIZE, shuffle=False)
    
    preds_s, targets_s = [], []
    with torch.no_grad():
        for X_b, y_b in loader_s:
            p = model(X_b.to(device)).cpu().numpy()
            preds_s.append(p.flatten())
            targets_s.append(y_b.numpy().flatten())
    
    preds_s = np.concatenate(preds_s)
    targets_s = np.concatenate(targets_s)
    subject_metrics[subj] = {
        "rmse": compute_rmse(preds_s, targets_s),
        "r2": compute_r2(preds_s, targets_s),
    }

for subj, m in subject_metrics.items():
    print(f"  {subj}: RMSE={m['rmse']:.3f} deg, R²={m['r2']:.3f}")

## Summary

| Metric | Value |
|--------|-------|
| Overall Test RMSE | (fill from above) |
| Overall Test R² | (fill from above) |
| P95 Latency | (fill from above) |
| Parameters | (fill from above) |

### Next Steps

- Swap LSTM encoder for Transformer/Informer (Phase 6)
- Fine-tune on amputee-specific data (domain adaptation)
- Integrate with impedance controller: `tau = K(theta_shadow - theta_current) + B(dtheta_shadow - dtheta_current)`